# 01 · Resume Extraction Agent

**Goal:** Turn a resume PDF into a validated `ResumeProfile` JSON that downstream agents (Gap Analysis, Deep Researcher, Job Hunter) can rely on.

This notebook implements **Stage 1 & Stage 2** from `Documentation/SkillGraph_ Resume Extraction and RAG Pipeline.pdf`:

1. **PDF → Markdown** via `pymupdf4llm` (fast, preserves section headers, OCR fallback for scanned PDFs).
2. **Markdown → structured JSON** via a **two-stage CoT prompt**:
   - Pass 1: Gemini reasons section-by-section and lists skills with evidence.
   - Pass 2: Gemini formats that reasoning into a strict Pydantic schema.
3. Wrap the whole thing as a **single LangGraph node** so it drops straight into the supervisor graph later.

### Why two-stage?
Per the docs: single-step extraction (prompt + schema in one call) consistently misses implied skills and produces lower-confidence level assignments. Two stages cost +1 Gemini call but the quality difference is significant because this profile feeds every downstream agent.

### Why Instructor-style validation?
Gemini's JSON output is inconsistent at the edges (missing fields, wrong types, occasionally wrapped in prose). We use LangChain's `with_structured_output(..., method="json_schema")` + a Pydantic retry wrapper to auto-reprompt on validation failure. `temperature=0` because extraction is not a creative task.


## 0. Setup

Install deps (once) and load API keys from `backend/.env`.


In [ ]:
# !pip install -q pymupdf4llm langchain langchain-google-genai langchain-core pydantic python-dotenv langgraph

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load API keys from the backend .env (GOOGLE_API_KEY, GROQ_API_KEY, TAVILY_API_KEY)
ENV_PATH = Path("../backend/.env").resolve()
load_dotenv(ENV_PATH)
assert os.getenv("GOOGLE_API_KEY"), f"GOOGLE_API_KEY missing — check {ENV_PATH}"
print("Loaded env from:", ENV_PATH)

## 1. Stage 1 — PDF → Markdown with `pymupdf4llm`

`pymupdf4llm.to_markdown()` preserves section headers as `##`, handles multi-column layouts, and drops OCR-fallback for scanned PDFs when extractable text is thin.


In [ ]:
import pymupdf4llm
import tempfile, os

def pdf_to_markdown(pdf_path: str, min_chars: int = 200) -> str:
    """Extract clean Markdown from a PDF. Returns empty string if unreadable."""
    md = pymupdf4llm.to_markdown(pdf_path)
    if len(md.strip()) < min_chars:
        # In production: fall back to OCR here (pymupdf4llm supports it via tesseract).
        # We keep the notebook lean and just warn.
        print(f"WARN: markdown is only {len(md)} chars — consider OCR fallback.")
    return md

In [ ]:
# A small synthetic resume we ship with the notebook so it runs without any upload.
SAMPLE_RESUME = '''# Priya Ramesh
priya.r@example.com · Bengaluru, India

## Summary
Final-year CS student with internship experience building ML pipelines and data APIs.

## Skills
- Languages: Python (3 yrs), SQL, TypeScript
- ML: PyTorch, scikit-learn, pandas
- Cloud & DevOps: Docker, AWS (S3, Lambda), GitHub Actions
- Tools: Git, Jupyter

## Experience
**ML Intern — Acme AI** (May 2025 – Aug 2025)
- Shipped a classification model (F1 0.87) over 2M rows using PyTorch.
- Authored FastAPI inference service deployed on AWS Lambda.

**Software Intern — Kite Labs** (Dec 2024 – Feb 2025)
- Built data-ingest DAG in Airflow; cut pipeline runtime 40%.

## Education
**B.Tech, Computer Science — IIT Madras** (2022 – 2026)

## Projects
- **SkillMap** — Next.js + Supabase app to visualise course prerequisites. https://github.com/priya/skillmap
- **TinyTorch** — Reimplemented autograd in 300 LOC. Pure NumPy.
'''

# Write the sample to disk as a fake .md (pymupdf4llm would give us this from a PDF).
# Swap this with `pdf_to_markdown("/path/to/your/resume.pdf")` when you have a real file.
resume_md = SAMPLE_RESUME
print(resume_md[:400], "...")

## 2. Stage 2 — Define the target schema

Per the docs, each skill carries `name`, `category`, `level` (beginner / intermediate / advanced), `evidence` (quote from resume), and optionally `years` (inferred from date ranges).


In [ ]:
from typing import List, Optional, Literal
from pydantic import BaseModel, Field

class Skill(BaseModel):
    name: str = Field(description="e.g. Python, PyTorch, AWS")
    category: Literal["Languages", "Frameworks", "Data", "ML", "Cloud & DevOps", "Tools", "Soft Skills"]
    level: Literal["beginner", "intermediate", "advanced"]
    evidence: str = Field(description="Short quote from the resume that justifies the level")
    years: Optional[float] = Field(None, description="Inferred from date ranges where present")

class Experience(BaseModel):
    role: str
    company: str
    start_date: Optional[str]
    end_date: Optional[str]
    bullets: List[str]

class Education(BaseModel):
    school: str
    degree: str
    start_date: Optional[str]
    end_date: Optional[str]

class Project(BaseModel):
    name: str
    description: Optional[str]
    tech: List[str] = Field(default_factory=list)
    link: Optional[str] = None

class ResumeProfile(BaseModel):
    name: str
    headline: Optional[str]
    email: Optional[str]
    location: Optional[str]
    summary: Optional[str]
    skills: List[Skill] = Field(default_factory=list)
    experience: List[Experience] = Field(default_factory=list)
    education: List[Education] = Field(default_factory=list)
    projects: List[Project] = Field(default_factory=list)

## 3. Stage 2a — Chain-of-Thought reasoning pass

Gemini walks through the resume section-by-section. We don't ask for JSON yet — we ask for **reasoning**. Output is plain text.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# temperature=0 for determinism on both stages
gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

cot_prompt = ChatPromptTemplate.from_template('''You are a technical recruiter reading a resume.

Walk through the resume SECTION BY SECTION. For each skill you find:
  - name the skill,
  - quote the line of evidence from the resume,
  - assign a level (beginner / intermediate / advanced) and justify it briefly,
  - if date ranges imply years of use, note them.

Do NOT output JSON yet. Reason in prose.

RESUME MARKDOWN:
---
{resume_md}
---''')

cot_chain = cot_prompt | gemini
reasoning = cot_chain.invoke({"resume_md": resume_md}).content
print(reasoning[:1500], "...")

## 4. Stage 2b — Structured formatting pass

Now feed the reasoning back to Gemini and ask for the strict `ResumeProfile` JSON. `with_structured_output` gives us automatic Pydantic validation; on validation failure LangChain will re-prompt (set `method="json_schema"` for strict mode with Gemini).


In [ ]:
format_prompt = ChatPromptTemplate.from_template('''Convert the following resume analysis into the required ResumeProfile schema.
Keep every `evidence` field as a verbatim phrase from the original resume.

RESUME MARKDOWN (for reference):
---
{resume_md}
---

ANALYSIS:
---
{reasoning}
---''')

structured_gemini = gemini.with_structured_output(ResumeProfile)
format_chain = format_prompt | structured_gemini

profile: ResumeProfile = format_chain.invoke({
    "resume_md": resume_md,
    "reasoning": reasoning,
})

import json
print(json.dumps(profile.model_dump(), indent=2)[:2000])

## 5. Wrap the two stages as a LangGraph node

In the full pipeline, Extraction is the first node in the supervisor graph. Here is the minimal subgraph — a single node that takes `{pdf_path | resume_md}` and produces `{profile}`.

Even though this is "just one node," we use LangGraph so the interface matches every downstream agent and state flows cleanly.


In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, START, END

class ExtractionState(TypedDict, total=False):
    pdf_path: Optional[str]
    resume_md: str
    reasoning: str
    profile: ResumeProfile

def node_load_markdown(state: ExtractionState) -> ExtractionState:
    if "resume_md" in state and state["resume_md"]:
        return {}  # already loaded
    return {"resume_md": pdf_to_markdown(state["pdf_path"])}

def node_cot(state: ExtractionState) -> ExtractionState:
    return {"reasoning": cot_chain.invoke({"resume_md": state["resume_md"]}).content}

def node_format(state: ExtractionState) -> ExtractionState:
    profile = format_chain.invoke({
        "resume_md": state["resume_md"],
        "reasoning": state["reasoning"],
    })
    return {"profile": profile}

g = StateGraph(ExtractionState)
g.add_node("load_markdown", node_load_markdown)
g.add_node("cot", node_cot)
g.add_node("format", node_format)
g.add_edge(START, "load_markdown")
g.add_edge("load_markdown", "cot")
g.add_edge("cot", "format")
g.add_edge("format", END)
extraction_graph = g.compile()

# Visualize
try:
    from IPython.display import Image, display
    display(Image(extraction_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(extraction_graph.get_graph().draw_mermaid())

In [ ]:
# Run the graph end-to-end on our sample resume markdown
final_state = extraction_graph.invoke({"resume_md": resume_md})
final_profile: ResumeProfile = final_state["profile"]

print(f"Name:     {final_profile.name}")
print(f"Headline: {final_profile.headline}")
print(f"Skills ({len(final_profile.skills)}):")
for s in final_profile.skills[:8]:
    print(f"  - {s.name:20s} [{s.level:12s}] {s.category:15s} — {s.evidence[:60]}")

## 6. Notes for the next engineer

- **Instructor library.** In production, wrap the LLM with `instructor` (native Gemini support) for Pydantic retry on validation errors. `with_structured_output` already does one retry; `instructor` gives you finer control and exposes the validation error to the LLM in the retry prompt.
- **Cache key.** Hash the sorted `ResumeProfile.model_dump_json()` + target role string as the Redis key — NOT the PDF bytes. Same resume re-uploaded should hit cache; role change should invalidate.
- **OCR fallback.** `pymupdf4llm.to_markdown(pdf, write_images=False)` + character-count check → fall through to Tesseract OCR. Trigger only when needed; don't pay OCR cost on every doc.
- **LLM routing.** Gemini is primary; on 429/5xx or rate limits, fall through to Groq (LLaMA-3-70B). The backend already has `app/utils/llm_factory.py` with both — lift those helpers into a shared `langgraph` node wrapper that routes automatically.
- **Downstream.** The `profile` output becomes the input to `02_gap_analysis_rag.ipynb` (the skills list specifically).
